# Experiment 4: Backward Chaining Engine with Askable Facts

**Aim:** To implement a backward chaining engine which is goal-driven reasoning for car-fault diagnosis using askable facts.

### Exercises
1. Add a **starter motor** fault using clicking sound and engine not cranking.
2. Change the order of hypotheses and observe the change in questions.
3. Add a **WHY** option to display the rule responsible for asking a question.

In [1]:
# Rules for car-fault diagnosis
rules = [
    (["lights_dim", "battery_old"], "battery_fault"),
    (["battery_ok", "alternator_warning"], "alternator_fault"),
    (["engine_cranks", "no_fuel"], "fuel_problem"),
    (["clicking_sound", "engine_not_cranking"], "starter_motor_fault"),
    (["engine_cranks", "no_spark"], "spark_plug_fault")
]

print("Rules:")
for conditions, conclusion in rules:
    print(" AND ".join(conditions), "->", conclusion)

Rules:
lights_dim AND battery_old -> battery_fault
battery_ok AND alternator_warning -> alternator_fault
engine_cranks AND no_fuel -> fuel_problem
clicking_sound AND engine_not_cranking -> starter_motor_fault
engine_cranks AND no_spark -> spark_plug_fault


In [2]:
# Questions for askable facts
questions = {
    "lights_dim": "Are the headlights dim? (yes/no)",
    "battery_old": "Is the battery old? (yes/no)",
    "battery_ok": "Is the battery working properly? (yes/no)",
    "alternator_warning": "Is the alternator warning light ON? (yes/no)",
    "engine_cranks": "Does the engine crank? (yes/no)",
    "no_fuel": "Is there no fuel reaching the engine? (yes/no)",
    "clicking_sound": "Do you hear a clicking sound when starting? (yes/no)",
    "engine_not_cranking": "Is the engine not cranking? (yes/no)",
    "no_spark": "Is there no spark? (yes/no)"
}

# Facts that are conclusions of rules
derived_facts = {conclusion for _, conclusion in rules}

print("Askable facts:")
for fact in questions:
    print("-", fact)

Askable facts:
- lights_dim
- battery_old
- battery_ok
- alternator_warning
- engine_cranks
- no_fuel
- clicking_sound
- engine_not_cranking
- no_spark


In [3]:
# Backward chaining engine with WHY option

facts = {}
asked = []

def find_rule(goal):
    for conditions, conclusion in rules:
        if conclusion == goal:
            return conditions
    return None

def backward_chain(goal, reason=None):
    # If already known, use the stored answer
    if goal in facts:
        return facts[goal]

    # If it is an askable fact, ask the user
    if goal not in derived_facts:
        while True:
            print("\n" + questions[goal])
            print("Type: yes / no / why")
            answer = input("> ").lower().strip()

            if answer == "yes":
                facts[goal] = True
                asked.append(goal)
                return True

            elif answer == "no":
                facts[goal] = False
                asked.append(goal)
                return False

            elif answer == "why":
                if reason:
                    print("WHY:", reason)
                else:
                    print("WHY: This fact is required as an initial fact for diagnosis.")
            else:
                print("Please enter yes, no, or why.")

    # Find a rule that can prove the goal
    conditions = find_rule(goal)

    if conditions is None:
        return False

    rule_text = " AND ".join(conditions) + " -> " + goal

    # All conditions must be true
    for condition in conditions:
        if not backward_chain(condition, rule_text):
            facts[goal] = False
            return False

    facts[goal] = True
    return True

In [4]:
# Hypotheses
hypotheses = [
    "battery_fault",
    "alternator_fault",
    "fuel_problem",
    "starter_motor_fault",
    "spark_plug_fault"
]

print("Hypothesis order:")
for i, h in enumerate(hypotheses, 1):
    print(i, h)

Hypothesis order:
1 battery_fault
2 alternator_fault
3 fuel_problem
4 starter_motor_fault
5 spark_plug_fault


In [5]:
# Diagnosis function

def diagnose():
    global facts, asked

    facts = {}
    asked = []

    print("\n--- CAR FAULT DIAGNOSIS ---")

    for hypothesis in hypotheses:
        if backward_chain(hypothesis):
            print("\nDiagnosis:", hypothesis)
            print("Facts asked:", asked)
            return hypothesis

    print("\nNo fault could be identified.")
    print("Facts asked:", asked)
    return None

## Exercise 1 — Starter Motor Fault

The new rule is:

**clicking_sound AND engine_not_cranking → starter_motor_fault**

Run the next cell and answer:

```text
clicking sound = yes
engine not cranking = yes
```

The expected diagnosis is `starter_motor_fault`.

In [6]:
diagnose()


--- CAR FAULT DIAGNOSIS ---

Are the headlights dim? (yes/no)
Type: yes / no / why


KeyboardInterrupt: Interrupted by user

## Exercise 2 — Change Hypothesis Order

Move `starter_motor_fault` to the first position. The engine will now try this hypothesis first, so the questions for the starter motor fault appear before questions for the other faults.

In [7]:
hypotheses = [
    "starter_motor_fault",
    "battery_fault",
    "alternator_fault",
    "fuel_problem",
    "spark_plug_fault"
]

print("New hypothesis order:")
for i, h in enumerate(hypotheses, 1):
    print(i, h)

New hypothesis order:
1 starter_motor_fault
2 battery_fault
3 alternator_fault
4 fuel_problem
5 spark_plug_fault


In [ ]:
diagnose()


--- CAR FAULT DIAGNOSIS ---

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking sound when starting? (yes/no)
Type: yes / no / why
Please enter yes, no, or why.

Do you hear a clicking so

## Exercise 3 — WHY Option

When the system asks a question, enter **why** instead of yes/no.

For example:

```text
Do you hear a clicking sound when starting?
Type: yes / no / why
> why

WHY: clicking_sound AND engine_not_cranking -> starter_motor_fault
```

This shows the rule responsible for asking the question.

In [ ]:
# Test WHY specifically with the starter motor hypothesis

hypotheses = ["starter_motor_fault"]

diagnose()

## Observe the Facts Asked

The following cell displays the facts asked during the latest diagnosis.

In [ ]:
print("Facts asked during diagnosis:")
for fact in asked:
    print("-", fact)

## Expected Observation

For the starter motor case:

- `clicking_sound` is asked.
- `engine_not_cranking` is asked.
- If both are **yes**, the correct fault `starter_motor_fault` is identified.
- Changing hypothesis order changes the order in which questions are asked.
- Selecting **WHY** displays the rule responsible for the question.

In [ ]:
# Final summary
print("=" * 55)
print("EXPERIMENT 4: BACKWARD CHAINING ENGINE")
print("=" * 55)
print("Starter motor rule added successfully.")
print("WHY option implemented successfully.")
print("Hypothesis ordering can be changed.")
print("Current facts asked:", asked)